# Validacao Robusta

Validacao temporal avancada para garantir estabilidade do modelo.

## Conteudo
1. Walk-Forward Validation
2. Backtesting por Periodo
3. Stability Tests

In [1]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from src.features.feature_engineering import FeatureEngineer
from src.models.evaluation import ModelEvaluator
print('Setup OK')

Setup OK


In [2]:
df = pd.read_parquet('../data/processed/processed_data.parquet')
fe = FeatureEngineer()
df_features = fe.create_all_features(df)
exclude_cols = ['timestamp', 'consumption_mw', 'region', 'holiday_name']
for col in df_features.columns:
    if pd.api.types.is_datetime64_any_dtype(df_features[col]) or df_features[col].dtype == 'object':
        if col not in exclude_cols: exclude_cols.append(col)
features = [c for c in df_features.columns if c not in exclude_cols]
df_sorted = df_features.sort_values('timestamp').reset_index(drop=True)
X = df_sorted[features].values
y = df_sorted['consumption_mw'].values
print(f'Data: {len(X):,} samples')

Creating features...
  - Temporal features
  - Lag features
  - Rolling features
  - Interaction features
  - Removed 240 rows with NaN
Data: 174,965 samples


## 1. Walk-Forward Validation

In [3]:
print('Walk-Forward Validation (simular producao)...')
tscv = TimeSeriesSplit(n_splits=5)
evaluator = ModelEvaluator()
wf_results = []

for fold, (train_idx, test_idx) in enumerate(tscv.split(X), 1):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    model = xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    metrics = evaluator.calculate_metrics(y_test, y_pred)
    
    wf_results.append({
        'Fold': fold,
        'Train_Size': len(X_train),
        'Test_Size': len(X_test),
        'MAPE': metrics['mape'],
        'R2': metrics['r2']
    })
    print(f'  Fold {fold}: MAPE={metrics["mape"]:.2f}% | R2={metrics["r2"]:.4f}')

df_wf = pd.DataFrame(wf_results)
print(f'\nMedia MAPE: {df_wf["MAPE"].mean():.2f}% +/- {df_wf["MAPE"].std():.2f}%')
print(f'Media R2: {df_wf["R2"].mean():.4f} +/- {df_wf["R2"].std():.4f}')

Walk-Forward Validation (simular producao)...
  Fold 1: MAPE=1.45% | R2=0.9987
  Fold 2: MAPE=1.23% | R2=0.9990
  Fold 3: MAPE=1.17% | R2=0.9992
  Fold 4: MAPE=1.15% | R2=0.9993
  Fold 5: MAPE=1.09% | R2=0.9993

Media MAPE: 1.22% +/- 0.14%
Media R2: 0.9991 +/- 0.0002


## 2. Backtesting por Estacao

In [4]:
df_sorted['month'] = df_sorted['timestamp'].dt.month
df_sorted['season'] = df_sorted['month'].apply(lambda m: 'Inverno' if m in [12,1,2] else 'Primavera' if m in [3,4,5] else 'Verao' if m in [6,7,8] else 'Outono')

print('\nBacktesting por Estacao...')
season_results = []

for season in ['Primavera', 'Verao', 'Outono', 'Inverno']:
    df_season = df_sorted[df_sorted['season'] == season]
    
    if len(df_season) < 1000:
        continue
    
    split = int(0.7 * len(df_season))
    X_train = df_season.iloc[:split][features].values
    y_train = df_season.iloc[:split]['consumption_mw'].values
    X_test = df_season.iloc[split:][features].values
    y_test = df_season.iloc[split:]['consumption_mw'].values
    
    model = xgb.XGBRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=42, n_jobs=-1, verbosity=0)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    metrics = evaluator.calculate_metrics(y_test, y_pred)
    
    season_results.append({
        'Season': season,
        'MAPE': metrics['mape'],
        'R2': metrics['r2']
    })
    print(f'  {season}: MAPE={metrics["mape"]:.2f}% | R2={metrics["r2"]:.4f}')

df_seasons = pd.DataFrame(season_results)


Backtesting por Estacao...
  Primavera: MAPE=1.36% | R2=0.9988
  Verao: MAPE=1.31% | R2=0.9989
  Outono: MAPE=1.39% | R2=0.9988
  Inverno: MAPE=1.45% | R2=0.9988


## 3. Stability Tests

In [5]:
print('\nStability Test (re-treinar 10x com diferentes seeds)...')
stability_results = []

for seed in range(42, 52):
    model = xgb.XGBRegressor(n_estimators=300, max_depth=8, learning_rate=0.05, random_state=seed, n_jobs=-1, verbosity=0)
    
    split = int(0.70 * len(X))
    model.fit(X[:split], y[:split])
    y_pred = model.predict(X[split:])
    metrics = evaluator.calculate_metrics(y[split:], y_pred)
    
    stability_results.append({'Seed': seed, 'MAPE': metrics['mape'], 'R2': metrics['r2']})

df_stability = pd.DataFrame(stability_results)
print(f'MAPE: {df_stability["MAPE"].mean():.2f}% +/- {df_stability["MAPE"].std():.3f}%')
print(f'R2: {df_stability["R2"].mean():.4f} +/- {df_stability["R2"].std():.5f}')
print(f'\nModelo e ESTAVEL (baixa variancia)!' if df_stability['MAPE'].std() < 0.5 else 'Modelo tem VARIANCIA alta')


Stability Test (re-treinar 10x com diferentes seeds)...
MAPE: 1.13% +/- 0.000%
R2: 0.9993 +/- 0.00000

Modelo e ESTAVEL (baixa variancia)!


## 4. Resumo

In [6]:
print('\n' + '='*80)
print('RESUMO - VALIDACAO ROBUSTA')
print('='*80)
print(f'\n1. Walk-Forward (5 folds): MAPE = {df_wf["MAPE"].mean():.2f}% +/- {df_wf["MAPE"].std():.2f}%')
print(f'2. Backtesting Sazonal: Performance varia por estacao')
for _, row in df_seasons.iterrows():
    print(f'     {row["Season"]:10s}: MAPE = {row["MAPE"]:.2f}%')
print(f'3. Stability (10 seeds): MAPE = {df_stability["MAPE"].mean():.2f}% +/- {df_stability["MAPE"].std():.3f}%')
print('\n'+ '='*80)
print('Validacao robusta completa!')
print('='*80)


RESUMO - VALIDACAO ROBUSTA

1. Walk-Forward (5 folds): MAPE = 1.22% +/- 0.14%
2. Backtesting Sazonal: Performance varia por estacao
     Primavera : MAPE = 1.36%
     Verao     : MAPE = 1.31%
     Outono    : MAPE = 1.39%
     Inverno   : MAPE = 1.45%
3. Stability (10 seeds): MAPE = 1.13% +/- 0.000%

Validacao robusta completa!
